# 🎯 [C202_M02] 전술 메시지(전문) 패킷 파싱, 크기 압축 및 헤더 정보 전처리 모듈 구현
### Google Colab 실습 환경 (지휘통제(C2) / Level 2)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/C202/c202_m02_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 지휘통제(C2) (Level 2) |
| **미션 ID** | C202_M02 |
| **미션 주제** | 전술 메시지(전문) 패킷 파싱, 크기 압축 및 헤더 정보 전처리 모듈 구현 |
| **소요 시간** | 약 2시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
러시아-우크라이나 전쟁에서는 전자전 재밍과 통신 장비 부족으로 협대역 무선망의 대역폭이 크게 줄어드는 상황이 잦았고, 이때 적 접촉 보고나 의무후송 요청 같은 긴급 전문이 행정 전문 뒤에 묶여 늦게 전달되는 문제가 지휘통제의 약점이 되었다. 본 프로젝트에서는 가상 협대역 링크(2.4kbps)와 가상 전술 전문을 코드로 직접 만들어, 전문 구조 정의, 패킷 파싱·압축 전처리, 긴급도 기반 우선순위 큐, 채널 상태에 따른 지능형 재전송, 링크 신뢰도 산출, 통신 관제 화면, 통신관 HITL 통제, 대역폭 급감 스트레스 시험, AAR 재조정, 최종 통합까지 메시지 우선순위 큐·재전송 프로토타입을 단계별로 구현한다. 최종 목표는 대역폭이 제한된 상황에서도 FLASH·IMMEDIATE 전문을 기한 안에 전달하고, 어떤 전문도 잃어버리지 않으며, 순서 조정의 최종 판단은 통신관이 내리도록 설계하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

## 📂 2. 산출물 디렉토리 체계 및 실습 가이드

본 과제의 미션별 산출물(`deliverables`)은 단순 스크립트를 넘어 실무 엔터프라이즈 및 국방 지휘통제체계에 즉시 배포할 수 있는 **표준 패키지 디렉토리 구조**를 따릅니다:

| 디렉토리 | 주요 산출물(Deliverables) | 역할 및 설명 |
| :--- | :--- | :--- |
| **`data/`** | `data/schema/c202_input_schema.json` | 입출력 데이터 표준 스키마 및 원천/정제 데이터셋 |
| **`src/`** | `src/preprocess/`, `src/models/`, `src/verification/`, `src/ui/`, `src/hitl/`, `src/main_c202.py` | 전처리, 우선순위 큐, 재전송 제어, 링크 신뢰도, 대시보드, HITL 등 핵심 파이썬 소스 코드 |
| **`docs/`** | `docs/c202_data_spec.md`, `docs/c202_hitl_guide.md`, `docs/c202_aar_report.md`, `docs/c202_final_report.md` | 데이터 규격서, HITL 가이드, AAR 분석 보고서, 최종 브리핑 등 기술/지휘관 문서 |
| **`configs/`**| `configs/c202_tuned_params.yaml` | AAR 피드백 기반 재조정 임계치 및 환경설정 파일 |
| **`tests/`** | `tests/c202_stress_test.py` | 대역폭 90% 급감 스트레스 시험 및 단위 테스트 스위트 |

> **💡 [Google Colab / 주피터 노트북 실습 팁]**
> - **방법 A. 자동 파일 저장 (권장):** 코드 셀 첫 줄에 `%%writefile src/preprocess/c202_preprocessor.py`와 같이 매직 명령어를 추가하여 실행하면, 노트북에서 작업한 코드가 해당 폴더의 파일로 자동 추출·저장됩니다.
> - **방법 B. 논리적 모듈 대응:** 별도 파일 추출 없이 노트북 셀 단위로 순차 실행하더라도 각 미션이 해당 모듈의 역할을 1:1로 수행하며, **미션 10(시스템 통합)**에서 전체 파이프라인이 유기적으로 완성되도록 설계되어 있습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 02. 전술 메시지(전문) 패킷 파싱, 크기 압축 및 헤더 정보 전처리 모듈 구현
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

전술 메시지 패킷 파싱, 헤더 압축 및 페이로드 데이터 정규화 전처리 모듈을 구현한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **잡음이 섞인 원시 전문 패킷을 파싱하고 긴급도 표기 정규화, CRC 검사, 중복 제거, 결측 시각 보정, 헤더 이진 압축, 본문 압축을 수행하는 전처리 모듈을 구현한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **긴급도 표기 혼재, 재전송으로 인한 중복 수신, 비트 오류, 전송 중 잘림, 미등록 송신국(스푸핑 의심) 패킷을 포함한 가상 원시 패킷 작성하여 원인을 분석하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/preprocess/c202_preprocessor.py 및 전처리 전후 데이터셋` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 전술 환경 특유의 노이즈(스푸핑, 반사파, 센서 단절 등) 제거 타당성

#### 미션 PASS 조건

- 정량 기준: 데이터 정제 처리 속도 1,000건/초 이상, 전처리 오류율 0%


- 진행 기준: 정제 파이프라인 자동화 및 단위 테스트 케이스 5종 통과


- 검증 방법: pytest 기반 전처리 정합성 테스트 및 결측치 보정율 확인


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""가상 전술 전문 패킷(텍스트 헤더+본문)을 파싱하고 긴급도 표기 정규화·CRC 검사·중복 제거·결측 시각 보정·헤더 이진 압축·본문 압축을 수행하는 전처리 모듈을 구현해 처리 속도와 단위 테스트 통과 여부를 확인한다."""
import struct
import time
import zlib
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 전술 전문(메시지) 생성기 (실제 부대·통신망·제원 정보 없음)
# 긴급도 4단계(FLASH/IMMEDIATE/PRIORITY/ROUTINE)와 전문 유형별 크기를 가진 전문을 만든다.
# ---------------------------------------------------------------------------
PREC = ["FLASH", "IMMEDIATE", "PRIORITY", "ROUTINE"]
RANK = {p: i for i, p in enumerate(PREC)}
DEADLINE = {"FLASH": 10, "IMMEDIATE": 30, "PRIORITY": 120, "ROUTINE": 600}   # 가상 전달 목표 시간(초)
MSG_TYPES = {   # 유형: (긴급도 후보, 확률, 본문 크기 범위 bytes, 발생 비율)
    "CONTACT": (["FLASH", "IMMEDIATE"], [0.7, 0.3], (120, 300), 0.08),
    "MEDEVAC": (["FLASH", "IMMEDIATE"], [0.4, 0.6], (150, 350), 0.05),
    "SITREP": (["IMMEDIATE", "PRIORITY"], [0.2, 0.8], (600, 2500), 0.20),
    "POSREP": (["PRIORITY", "ROUTINE"], [0.5, 0.5], (60, 120), 0.35),
    "LOGREQ": (["PRIORITY", "ROUTINE"], [0.4, 0.6], (300, 1200), 0.14),
    "ADMIN": (["ROUTINE"], [1.0], (1000, 5000), 0.18),
}


def make_messages(duration=600, rate=0.12, seed=7, t_start=0.0, prefix="M"):
    """duration초 동안 평균 rate건/초로 발생하는 가상 전문 목록."""
    rng = np.random.default_rng(seed)
    n = int(rng.poisson(rate * duration))
    times = np.sort(rng.uniform(t_start, t_start + duration, n))
    types = list(MSG_TYPES)
    msgs = []
    for i, t0 in enumerate(times):
        ty = types[rng.choice(len(types), p=[MSG_TYPES[k][3] for k in types])]
        cands, p, (lo, hi), _ = MSG_TYPES[ty]
        msgs.append({"id": f"{prefix}{i:04d}", "t0": float(t0), "type": ty, "prec": str(rng.choice(cands, p=p)),
                     "size": int(rng.integers(lo, hi + 1)), "src": f"U{int(rng.integers(1, 9)):02d}"})
    return msgs


class Link:
    """[교육용가상자료] 가상 협대역 무선 링크(기본 2.4kbps). 초 단위 상태(good/bad/jam)에 따라
    비트오류율이 바뀌고(Gilbert-Elliott 모델), 구간별 대역폭 감소도 줄 수 있다."""
    BER = {"good": 1e-5, "bad": 2e-4, "jam": 1.2e-3}

    def __init__(self, horizon=4000, bw_bps=2400, seed=1, p_gb=0.02, p_bg=0.15, jam=(), bw_drop=(), rtt=(1.2, 0.4)):
        rng = np.random.default_rng(seed)
        st, s = [], "good"
        for _ in range(int(horizon) + 1):
            s = ("bad" if rng.random() < p_gb else "good") if s == "good" else ("good" if rng.random() < p_bg else "bad")
            st.append(s)
        for a, b in jam:
            for k in range(int(a), min(int(b), len(st))):
                st[k] = "jam"
        self.states, self.bw_bps, self.bw_drop, self.rtt_par = st, bw_bps, list(bw_drop), rtt

    def state(self, t):
        return self.states[min(int(t), len(self.states) - 1)]

    def bw(self, t):
        f = 1.0
        for a, b, fac in self.bw_drop:
            if a <= t < b:
                f = fac
        return self.bw_bps * f

    def p_loss(self, nbytes, t):
        return 1 - (1 - self.BER[self.state(t)]) ** (8 * nbytes)

    def rtt(self, rng):
        return self.rtt_par[0] + rng.exponential(self.rtt_par[1])
# ---------------------------------------------------------------------------

PREC_ALIAS = {"Z": "FLASH", "FLASH": "FLASH", "O": "IMMEDIATE", "IMMED": "IMMEDIATE", "IMMEDIATE": "IMMEDIATE",
              "P": "PRIORITY", "PRIORITY": "PRIORITY", "R": "ROUTINE", "ROUTINE": "ROUTINE"}
ROSTER = {f"U{i:02d}" for i in range(1, 9)}                  # 가상 망 가입국 목록
TYPES = list(MSG_TYPES)


def make_body(m, rng):
    """[교육용가상자료] 'KEY=VALUE;' 형식의 정형 본문(실제 전문 양식 아님)."""
    lines, k = [], 0
    while sum(map(len, lines)) < m["size"]:
        lines.append(f"L{k:02d}={m['type']} UNIT {m['src']} ITEM {int(rng.integers(1, 40)):02d} QTY {int(rng.integers(0, 99)):02d} STAT OK;")
        k += 1
    return "".join(lines)[:m["size"]]


def make_raw_packets(msgs, seed=5):
    """전술 환경 잡음을 넣은 원시 패킷 줄: 긴급도 표기 혼재, 중복 수신, 비트 오류(CRC 불일치), 시각 누락, 잘림, 미등록 송신국."""
    rng = np.random.default_rng(seed)
    out = []
    for m in msgs:
        body = make_body(m, rng)
        prec = rng.choice([PREC[RANK[m["prec"]]], PREC[RANK[m["prec"]]].lower(), "ZOPR"[RANK[m["prec"]]], " " + "ZOPR"[RANK[m["prec"]]] + " "])
        dtg = "" if rng.random() < 0.05 else f"{m['t0']:.1f}"
        src = m["src"] if rng.random() > 0.02 else "X77"
        line = f"ZCZC|ID={m['id']}|PREC={prec}|TYPE={m['type']}|SRC={src}|DTG={dtg}|LEN={len(body)}|CRC={zlib.crc32(body.encode()):08x}|BODY={body}"
        r = rng.random()
        if r < 0.03:                                            # 비트 오류: 본문 일부 변형
            line = line[:-5] + "#" + line[-4:]
        elif r < 0.05:                                          # 전송 중 잘림
            line = line[:int(len(line) * 0.4)]
        out.append(line)
        if rng.random() < 0.06:                                 # 재전송으로 인한 중복 수신
            out.append(line)
    return out


class ParseError(ValueError):
    pass


def parse(line):
    if not line.startswith("ZCZC|"):
        raise ParseError("시작 표지 없음")
    kv = dict(p.split("=", 1) for p in line[5:].split("|") if "=" in p)
    for k in ("ID", "PREC", "TYPE", "SRC", "DTG", "LEN", "CRC", "BODY"):
        if k not in kv:
            raise ParseError(f"필드 누락({k})")
    prec = PREC_ALIAS.get(kv["PREC"].strip().upper())
    if prec is None:
        raise ParseError("긴급도 표기 오류")
    if kv["TYPE"] not in MSG_TYPES:
        raise ParseError("전문 유형 오류")
    if kv["SRC"] not in ROSTER:
        raise ParseError("미등록 송신국(스푸핑 의심)")
    if int(kv["LEN"]) != len(kv["BODY"]) or f"{zlib.crc32(kv['BODY'].encode()):08x}" != kv["CRC"]:
        raise ParseError("CRC/길이 불일치(비트 오류)")
    return {"id": kv["ID"], "prec": prec, "type": kv["TYPE"], "src": kv["SRC"],
            "dtg": float(kv["DTG"]) if kv["DTG"] else np.nan, "body": kv["BODY"], "hdr_bytes": len(line) - len(kv["BODY"])}


def pack_header(r):
    """헤더 이진 압축: ID(2) + 긴급도·유형·송신국(2) + 시각(4, 0.1초) + 길이(2) + CRC(4) = 14bytes."""
    flags = RANK[r["prec"]] << 12 | TYPES.index(r["type"]) << 8 | int(r["src"][1:])
    return struct.pack(">HHIHI", int(r["id"][1:]), flags, int(round(r["dtg"] * 10)), len(r["body"]), zlib.crc32(r["body"].encode()))


def unpack_header(b):
    i, flags, dtg, ln, crc = struct.unpack(">HHIHI", b)
    return {"id": f"M{i:04d}", "prec": PREC[flags >> 12], "type": TYPES[(flags >> 8) & 0xF], "src": f"U{flags & 0xFF:02d}", "dtg": dtg / 10, "len": ln}


ZLEVEL = {1200: 9, 2400: 9, 9600: 6, 16000: 1}                  # 대역폭이 좁을수록 압축을 강하게


def preprocess(lines, bw_bps=2400):
    good, quarantine, seen = [], [], set()
    for ln in lines:
        try:
            r = parse(ln)
        except (ParseError, ValueError) as e:
            quarantine.append(str(e)); continue
        if r["id"] in seen:
            quarantine.append("중복 수신(제거)"); continue
        seen.add(r["id"]); good.append(r)
    good.sort(key=lambda r: int(r["id"][1:]))
    dtg = pd.Series([r["dtg"] for r in good], dtype=float)
    miss = int(dtg.isna().sum())
    filled = dtg.interpolate(limit_direction="both")              # 앞뒤 전문 시각으로 선형 보간
    for r, v in zip(good, filled):
        r["dtg_filled"] = bool(np.isnan(r["dtg"])); r["dtg"] = float(v)
        r["hdr_packed"] = pack_header(r)
        r["body_z"] = zlib.compress(r["body"].encode(), ZLEVEL[bw_bps])
    return good, quarantine, miss


msgs = make_messages(1200, 0.21, seed=7)
lines = make_raw_packets(msgs)
t0 = time.perf_counter()
good, quarantine, miss = preprocess(lines)
dt = time.perf_counter() - t0
print(f"원시 패킷 {len(lines)}줄 -> 정상 {len(good)}건, 격리 {len(quarantine)}건 / 처리 속도 {len(lines) / dt:,.0f}건/초 (목표 1,000건/초 이상)")
print("처리 중 예외로 멈춘 건수 0건(오류율 0%) - 비정상 패킷은 사유와 함께 격리")
print("격리 사유:", pd.Series(quarantine).value_counts().to_dict())
print(f"DTG 결측 {miss}건 중 보정 {sum(r['dtg_filled'] for r in good)}건 (보정율 {sum(r['dtg_filled'] for r in good) / max(miss, 1):.0%})")

# 전처리 전후 크기 비교(전문 유형별 평균 bytes)
df = pd.DataFrame([{"유형": r["type"], "원본 헤더": r["hdr_bytes"], "압축 헤더": len(r["hdr_packed"]),
                    "원본 본문": len(r["body"]), "압축 본문": len(r["body_z"])} for r in good])
tab = df.groupby("유형").mean().round(0).astype(int)
tab["전체 절감률"] = (1 - (tab["압축 헤더"] + tab["압축 본문"]) / (tab["원본 헤더"] + tab["원본 본문"])).round(3)
print("\n[전처리 전후 평균 크기(bytes)]")
print(tab.to_string())
tot0, tot1 = (df["원본 헤더"] + df["원본 본문"]).sum(), (df["압축 헤더"] + df["압축 본문"]).sum()
print(f"전체 {tot0:,}B -> {tot1:,}B, 2.4kbps 기준 전송시간 {tot0 * 8 / 2400 / 60:.1f}분 -> {tot1 * 8 / 2400 / 60:.1f}분")
print("상황별 압축 수준(대역폭 bps: zlib 수준):", ZLEVEL)
print("주의: 가상 본문은 정형 반복 문구라 압축률이 높게 나온다. 자유 서술 본문은 절감률이 이보다 낮다.")

# 단위 테스트 5종(+1): 조건을 알고 있는 시험용 패킷으로 확인
def mk(prec="Z", src="U01", body="L00=TEST UNIT U01;", crc=None):
    return f"ZCZC|ID=M9999|PREC={prec}|TYPE=CONTACT|SRC={src}|DTG=1.0|LEN={len(body)}|CRC={crc or f'{zlib.crc32(body.encode()):08x}'}|BODY={body}"


def raises(line):
    try:
        parse(line)
        return False
    except ParseError:
        return True


r0 = good[0]
tests = {
    "정상 패킷 파싱": parse(mk())["prec"] == "FLASH",
    "긴급도 표기 정규화(' o ' -> IMMEDIATE)": parse(mk(prec=" o "))["prec"] == "IMMEDIATE",
    "CRC 불일치 검출": raises(mk(crc="00000000")),
    "미등록 송신국 격리": raises(mk(src="X77")),
    "중복 수신 제거": len(preprocess([mk(), mk()])[0]) == 1 and len({r["id"] for r in good}) == len(good),
    "결측 시각 보간 + 헤더 압축 왕복 일치": not any(np.isnan(r["dtg"]) for r in good) and unpack_header(r0["hdr_packed"])["id"] == r0["id"],
}
for k, v in tests.items():
    print(f"[{'PASS' if v else 'FAIL'}] {k}")
print(f"단위 테스트 {sum(tests.values())}/{len(tests)} 통과")


<details>
<summary><b>💡 [참고] 미션 02 예상 실행 결과 (Output)</b></summary>

```text
원시 패킷 278줄 -> 정상 241건, 격리 37건 / 처리 속도 19,781건/초 (목표 1,000건/초 이상)
처리 중 예외로 멈춘 건수 0건(오류율 0%) - 비정상 패킷은 사유와 함께 격리
격리 사유: {'중복 수신(제거)': 18, 'CRC/길이 불일치(비트 오류)': 11, '미등록 송신국(스푸핑 의심)': 5, '필드 누락(BODY)': 3}
DTG 결측 15건 중 보정 15건 (보정율 100%)

[전처리 전후 평균 크기(bytes)]
         원본 헤더  압축 헤더  원본 본문  압축 본문  전체 절감률
유형                                         
ADMIN       80     14   3506    555   0.841
CONTACT     81     14    227     93   0.653
LOGREQ      81     14    760    178   0.772
MEDEVAC     80     14    206     91   0.633
POSREP      80     14     89     63   0.544
SITREP      82     14   1666    303   0.819
전체 276,195B -> 54,134B, 2.4kbps 기준 전송시간 15.3분 -> 3.0분
상황별 압축 수준(대역폭 bps: zlib 수준): {1200: 9, 2400: 9, 9600: 6, 16000: 1}
주의: 가상 본문은 정형 반복 문구라 압축률이 높게 나온다. 자유 서술 본문은 절감률이 이보다 낮다.
[PASS] 정상 패킷 파싱
[PASS] 긴급도 표기 정규화(' o ' -> IMMEDIATE)
[PASS] CRC 불일치 검출
[PASS] 미등록 송신국 격리
[PASS] 중복 수신 제거
[PASS] 결측 시각 보간 + 헤더 압축 왕복 일치
단위 테스트 6/6 통과
```
</details>
